# Week 4, Day 4: Open-source contenders in one app

**Lesson:** [week4/day4.ipynb](../../../week4/day4.ipynb), *Code generator, continued*. **About 40 minutes.**

**What you practice here**

- Reaching open-source models through the same `OpenAI` class: Ollama on your machine, with the key `"ollama"`, and Groq and OpenRouter hosted, each with its own key.
- One function for every model: pick the client by model name from a dict, as the lesson's `clients` does, ask only OpenAI's reasoning models for reasoning effort, and return the code so an app can show it.
- Capturing what a Python program prints, so an app can show it: `exec` in a fresh namespace, with the standard output sent to a string buffer until the program is done.
- Wiring a Gradio app whose button calls your function with the dropdown's model and the textbox's code, in Try it for real.

**How this notebook works.** Run the first code cell once. Then, for each part below, write it in its cell, run that cell, and run the check under it. The checks hand your code stand-ins shaped like the real client and libraries: they record what your code sends and answer with canned replies, so nothing is called and nothing is spent. Write from memory, and look back at the lesson only when you are stuck.

When every check passes, save the notebook, run the checker and push, as in every session. The tracker then marks the lesson done.

In [ ]:
# Run this cell first. It finds plan/check.py in your fork and loads this exercise's checks.
import pathlib, sys
ROOT = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "plan" / "check.py").is_file()), None)
if ROOT:
    sys.path.insert(0, str(ROOT / "plan"))
    try:
        from check import exercise
    except ImportError:
        raise ImportError("your plan/check.py is older than this notebook: run git pull, or copy the new one from the tracker page") from None
    check = exercise("week4/day4", globals())
else:  # in Colab, or outside your fork: write and try the parts here, and run the checks in your codespace
    def check(part=None):
        print("The checks are off here, as plan/check.py is not in a folder above this notebook. Run them in your codespace.")
    print("plan/check.py was not found above this notebook, so the checks are off here, as in Colab.")

## The task

The lesson ports the same program with open-source models, small ones on your machine through Ollama and bigger ones
hosted on Groq and OpenRouter, and puts the comparison in a Gradio app: pick a model, press Convert, and read the
C++. Build those pieces here for your Royal Rumble simulator from Day 3.

`open_source_clients` makes the clients. The function that calls a model, `port_with`, takes a dict from model name
to client, like the lesson's `clients`, as its first parameter: it picks the client for the model it is asked to use
and makes the Chat Completions call on it. The checks hand it stand-ins shaped like the real client, so nothing is
spent. `run_python` runs a Python program and returns what it printed, for the app's Run button. In Try it for real,
you load your Day 3 notebook and wire the app.

## What each call should give

```python
clients = open_source_clients({"GROQ_API_KEY": "gsk_..."})
sorted(clients)                # ['groq', 'ollama']
clients["ollama"].base_url     # URL('http://localhost:11434/v1/')

by_model = {"qwen2.5-coder:1.5b": clients["ollama"], "openai/gpt-oss-120b": clients["groq"]}
port_with(by_model, "openai/gpt-oss-120b", messages)
# makes one request, on the Groq client, with the model openai/gpt-oss-120b, those messages and no reasoning
# effort; a reply of '```cpp\n#include <cstdio>\n...\n```' comes back as '#include <cstdio>\n...'

run_python('print("Total:", 93173 + 101763)')
# 'Total: 194936\n'
run_python("1 / 0")
# 'Error: division by zero'
```

In [ ]:
# The imports for this exercise. Add more here if you need them: the checks have the standard library, openai, pydantic, dotenv, numpy, bs4 and tiktoken.
import io
import sys

from openai import OpenAI

### 1 · `open_source_clients(env)`

Makes one `OpenAI` client for each open-source provider you can reach, and returns them in a dict keyed by provider:
Ollama always, and Groq and OpenRouter when their keys are set. `env` is a mapping of environment variables, as on
Day 3: read the keys from `env` only. Make:

- `"ollama"`, always: a client at `http://localhost:11434/v1` with the key `"ollama"`, as Ollama runs on your machine
  and needs no real key;
- `"groq"` and `"openrouter"`: a client at that provider's base URL, with its key from `env` (`GROQ_API_KEY` and
  `OPENROUTER_API_KEY`). The base URLs, from the lesson, are `https://api.groq.com/openai/v1` and
  `https://openrouter.ai/api/v1`.

Leave out Groq or OpenRouter when its key is missing or empty. The checks use the real `OpenAI` here and read its base
URL and key.

<details><summary>Hint</summary>

```python
OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
OpenAI(base_url="https://api.groq.com/openai/v1", api_key=env["GROQ_API_KEY"])
```

</details>

In [ ]:
def open_source_clients(env):
    # Write open_source_clients here, as described above.
    raise NotImplementedError("open_source_clients is not written yet")

In [ ]:
check("open_source_clients")

### 2 · `port_with(clients, model, messages)`

Asks one model for the port, through the client that serves it, and returns the code. `clients` is a dict from model
name to client, like the lesson's `clients`, such as `{"qwen2.5-coder:1.5b": ollama, "openai/gpt-oss-120b": groq}`,
and `messages` are the messages to send, such as your Day 3 `messages_for(...)`. Make one Chat Completions call, on
the client for `model` and no other, with the model `model` and those messages, passing everything by name.

- Set the reasoning effort to `"high"` only for OpenAI's own reasoning models. Keep their names in a set, as the
  lesson's `{"gpt-5"}` does, and add `gpt-5-mini` or `gpt-5-nano` if you use them.
- Every other model gets none (`None`), including the open-weight gpt-oss models such as `openai/gpt-oss-120b`:
  their names contain "gpt" too, which is why the lesson stops testing for `'gpt' in model` here.

Return the code from the reply's text as Day 3's `port` does: drop every line that starts with three backticks,
whatever follows them (`cpp`, `rust` or nothing), and remove the whitespace at both ends.

<details><summary>Hint</summary>

```python
client = clients[model]
reasoning_effort = "high" if model in OPENAI_REASONING_MODELS else None
response = client.chat.completions.create(model=model, messages=messages, reasoning_effort=reasoning_effort)
```

`OPENAI_REASONING_MODELS` is your set of names, written above the function.

</details>

In [ ]:
def port_with(clients, model, messages):
    # Write port_with here, as described above.
    raise NotImplementedError("port_with is not written yet")

In [ ]:
check("port_with")

### 3 · `run_python(code)`

Runs a Python program and returns what it printed, as the lesson's `run_python` does for the app's Run button. Run the
code with `exec`, giving it a new dict for each run that holds only `__builtins__`, as its globals alone: with a
separate dict for the locals, a program whose functions call each other fails. While it runs, send what it prints
into a string buffer, as the lesson does by swapping `sys.stdout`, and return the text it holds.

If the program raises an exception, return `"Error: "` followed by the exception's message, as the lesson does.
Either way, put the real standard output back before you return.

<details><summary>Hint</summary>

```python
buffer = io.StringIO()
old_stdout, sys.stdout = sys.stdout, buffer
try:
    exec(code, {"__builtins__": __builtins__})
    ...
finally:
    sys.stdout = old_stdout
```

`contextlib.redirect_stdout(buffer)`, used in a `with` statement, does the swap and the restore for you.

</details>

In [ ]:
def run_python(code):
    # Write run_python here, as described above.
    raise NotImplementedError("run_python is not written yet")

In [ ]:
check("run_python")

## Check everything

The cell below runs all 8 checks again. When they all pass, save this notebook (Ctrl+S, or Cmd+S on a Mac), then run the checker and push, as in every session:

```bash
uv run python plan/check.py
git add -A
git commit -m "Week 4, Day 4 exercise"
git push
```

The checker runs the same checks on the saved notebook, and the tracker marks the lesson done. It reads only your imports, functions, classes and constants, so the experiments in your other cells never get in the way.

In [ ]:
check()

## Try it for real

Now run what you wrote on the real thing. Write the steps in the cell at the end; they may call a model with your key, and the checks leave them out.

Ollama is free but must be running, with `qwen2.5-coder:1.5b` pulled (`ollama pull qwen2.5-coder:1.5b`, about 1 GB).
On a codespace's two CPUs, expect a minute or more per port. Reach the gpt-oss models through Groq, which has a free
tier, as `openai/gpt-oss-20b` and `openai/gpt-oss-120b`: the lesson's `gpt-oss:20b` on Ollama is a 14 GB download that
needs about 16 GB of memory, more than a standard codespace has. OpenRouter costs little for
`qwen/qwen3-coder-30b-a3b-instruct`.

1. Load your keys with `load_dotenv(override=True)`. Put the course's `week4` folder on the path with
   `sys.path.append(str(ROOT / "week4"))` and import `retrieve_system_info` from `system_info`. Then load your Day 3
   notebook with `day3 = functions_of("week4/day3")`, after `from check import functions_of`.
2. Build the lesson's `clients` dict from model name to client, for the models you can reach: `gpt-5-nano` on
   `day3.frontier_clients(os.environ)["openai"]`, `qwen2.5-coder:1.5b` on `open_source_clients(os.environ)["ollama"]`,
   `openai/gpt-oss-20b` and `openai/gpt-oss-120b` on Groq, and `qwen/qwen3-coder-30b-a3b-instruct` on OpenRouter.
   Then set `system_info = retrieve_system_info()` and your `compile_command`, as on Day 3.
3. Write `convert(model, python)`, the function the Convert button calls with the dropdown's and the textbox's values.
   It returns `port_with(clients, model, day3.messages_for(python, system_info, compile_command))`.
4. Build the lesson's app with `gr.Blocks()`: a Python textbox holding `day3.RUMBLE`, a C++ textbox, a dropdown of
   your models, a "Convert code" button wired with `.click(convert, inputs=[model, python], outputs=[cpp])`, and a
   "Run Python" button wired to `run_python`. Launch it and convert with two or three models.
5. Optional: compile and run a port as on Day 3, and note which models are correct and how fast they are.

<details><summary>Stuck? One way to write it</summary>

```python
import os
import sys

import gradio as gr
from dotenv import load_dotenv

from check import functions_of

load_dotenv(override=True)
sys.path.append(str(ROOT / "week4"))  # the course's week4 folder, for its system_info helper
from system_info import retrieve_system_info

day3 = functions_of("week4/day3")  # your Day 3 notebook, as saved

providers = {**day3.frontier_clients(os.environ), **open_source_clients(os.environ)}
wanted = {"gpt-5-nano": "openai", "qwen2.5-coder:1.5b": "ollama", "openai/gpt-oss-20b": "groq",
          "openai/gpt-oss-120b": "groq", "qwen/qwen3-coder-30b-a3b-instruct": "openrouter"}
clients = {model: providers[name] for model, name in wanted.items() if name in providers}
system_info = retrieve_system_info()
compile_command = ["g++", "-std=c++17", "-O3", "-march=native", "main.cpp", "-o", "main"]  # yours, as on Day 3


def convert(model, python):
    return port_with(clients, model, day3.messages_for(python, system_info, compile_command))


with gr.Blocks() as ui:
    with gr.Row():
        python = gr.Textbox(label="Python code:", lines=28, value=day3.RUMBLE)
        cpp = gr.Textbox(label="C++ code:", lines=28)
    with gr.Row():
        model = gr.Dropdown(list(clients), label="Select model", value=next(iter(clients)))
        convert_button = gr.Button("Convert code")
        run_button = gr.Button("Run Python")
    python_out = gr.Textbox(label="Python result", lines=4)
    convert_button.click(convert, inputs=[model, python], outputs=[cpp])
    run_button.click(run_python, inputs=[python], outputs=[python_out])

ui.launch(inbrowser=True)
```

</details>

In [ ]:
# Try it for real: write the steps above here, in as many cells as you like. This cell may call a model with your key; the checks leave it out.